# Data loading from drive

In [8]:
#Load data set from the google drive
from google.colab import drive
import pathlib

drive.mount('/content/drive')
!ls '/content/drive/MyDrive/MSC/DataSet/phm/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
phm_test.csv  phm_train.csv  phm_train.gsheet


# Common Imports

In [47]:
# Imports
import pandas as pd
import nltk
from nltk.corpus import stopwords
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer  # to encode text to int
from tensorflow.keras.preprocessing.sequence import pad_sequences   # to do padding or truncating
from tensorflow.keras.models import Sequential     # the model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout # layers of the architecture
from tensorflow.keras.callbacks import ModelCheckpoint   # save model
from tensorflow.keras.models import load_model   # load saved model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras import backend as K
import re


# Function definition for all models




In [48]:
# Function load data from drive as csv
def load_from_csv():
    train_data = pd.read_csv('/content/drive/MyDrive/MSC/DataSet/phm/phm_train.csv')
    test_data = pd.read_csv('/content/drive/MyDrive/MSC/DataSet/phm/phm_test.csv')

    print('\nData loaded..')

    return train_data, test_data


# Function for pre process data
def preprocess_dataset(tweet_data):
    x_data = tweet_data['tweet']       # Reviews/Input  --> Object
    y_data = tweet_data['label']    # Sentiment/Output  --> Int

    # PRE-PROCESS REVIEW
    nltk.download('stopwords')
    english_stops = set(stopwords.words('english'))

    x_data = x_data.replace({'<.*?>': ''}, regex = True)          # remove html tag
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
    x_data = x_data.apply(lambda tweet: [w for w in tweet.split() if w not in english_stops])  # remove stop words
    x_data = x_data.apply(lambda tweet: [w.lower() for w in tweet])   # lower case

    # y_data already encoded as 0 and 1 ( int values)
    print('\nData pre-process completed..')

    return x_data, y_data

# Function for get taining and testing data
def get_dataset(train_data, test_data):
    # remove tweet_id from df
    train_data = train_data.drop('tweet_id', axis=1)
    test_data = test_data.drop('tweet_id', axis=1)

    train_data = train_data[train_data['label'].isin([0, 1])]
    test_data = test_data[test_data['label'].isin([0, 1])]

    x_train, y_train = preprocess_dataset(train_data)
    x_test, y_test = preprocess_dataset(test_data)

    return x_train, y_train, x_test, y_test

# Function for getting the maximum tweet length
def get_max_length(x_train):
    tweet_length = []
    for tweet in x_train:
        length = len(tweet)
        tweet_length.append(length)
    #print('Max tweet length: ', np.max(tweet_length))
    #print('Min tweet length: ', np.min(tweet_length))
    #print('Mean tweet length:', np.mean(tweet_length))
    return int(np.ceil(np.mean(tweet_length)))

# Function for tokenize data
def tokenize_data(x_train, x_test):
    token = Tokenizer(lower=False)
    #print(x_train[9073]) # Check for 'how can a being be doing xanax' after stop wording it becomes 'xanax'
    token.fit_on_texts(x_train)
    x_train_seq = token.texts_to_sequences(x_train)
    #print(x_train[9073]) # 'xanax' token is 10
    x_test_seq = token.texts_to_sequences(x_test)

    max_length = get_max_length(x_train)

    x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='post', truncating='post')
    x_test_pad = pad_sequences(x_test_seq, maxlen=max_length, padding='post', truncating='post')

    total_words = len(token.word_index) + 1

    print('Maximum tweet length: ', max_length)
    print('Total words: ', total_words)

    return x_train_pad, x_test_pad, max_length, total_words

# Function for test the model
def test_model(x_test,y_test):
    print("\n\nTesting ---------------------------------------------------------------------")
    # Test the model
    y_pred = model.predict(x_test)
    y_pred = np.round(y_pred).astype(int)

    # Get accurately predicted count
    accurate_count = 0
    for i, y in enumerate(y_test):
        if y == y_pred[i]:
            accurate_count += 1

    print('\nCorrect Prediction: {}'.format(accurate_count))
    print('Wrong Prediction: {}'.format(len(y_pred) - accurate_count))
    accuracy = accurate_count/len(y_pred)*100
    print('Accuracy: {}'.format(accuracy))

    return accuracy


# Function for Trains and evaluates a model multiple times to compute average accuracy.
def evaluate_model_repeatedly(model, x_train, y_train, x_test, y_test, n_runs=5, batch_size=128, epochs=5, checkpoint=None):
    accuracies = []

    for run in range(n_runs):
        print(f"\n\nRun {run+1}/{n_runs} ----------------------------------------------------")
        # Reset model weights
        for layer in model.layers:
            if hasattr(layer, 'kernel_initializer'):
                # Get the initialization operation
                initial_weights = layer.get_weights()
                new_weights = []
                for w in initial_weights:
                    if hasattr(w, 'numpy'):  # For eager tensors
                        new_weights.append(w.numpy())
                    else:
                        new_weights.append(w)
                layer.set_weights(new_weights)


        # Train the model
        print("\n\nTraining ----------------------------------------------------------------")
        model.fit(x_train, y_train,batch_size=batch_size,epochs=epochs,callbacks=[checkpoint] if checkpoint else None)

        # Evaluate and store accuracy
        accuracy = test_model(x_test, y_test)  # Assumes test_model() returns a float
        accuracies.append(accuracy)

    # Compute statistics
    average_accuracy = np.mean(accuracies)
    std_dev = np.std(accuracies)

    return {
        "average_accuracy": average_accuracy,
        "std_dev": std_dev,
        "all_accuracies": accuracies,
    }

# Building AI model - LSTM

In [49]:
# Load data
train_data, test_data = load_from_csv()
# Get preprocessed data
x_train, y_train, x_test, y_test = get_dataset(train_data, test_data)
# Tokenizing data
x_train, x_test, max_length, total_words = tokenize_data(x_train, x_test)

# Build LSTM model
EMBED_DIM = 32
LSTM_OUT = 64

# define the model
model = Sequential()
model.add(Embedding(total_words, EMBED_DIM, input_length = max_length))
model.add(LSTM(LSTM_OUT))
model.add(Dense(1, activation='sigmoid'))

# Build the model
model.build(input_shape=(None, max_length))
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

# Train and test the model
checkpoint = ModelCheckpoint(
    'models/LSTM.h5',
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)

print('\nModel Summery ---------------------------------------------------------')
print(model.summary())
print('\n')

# Fit the model
# model.fit(x_train, y_train, batch_size = 128, epochs = 5, callbacks=[checkpoint])
# Test the model
# accuracy = test_model(x_test,y_test)

# Get the average accuracy
n_runs = 5  # Number of times to train and test
batch_size = 128
epochs = 5

# Example usage
results = evaluate_model_repeatedly(model=model, x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test, n_runs=n_runs, batch_size=batch_size, epochs=epochs, checkpoint=checkpoint)

# Access results
print('\n\n\nFinal Results -----------------------------------------------------')
average_accuracy_LSTM_1 = results["average_accuracy"]
print('\nAverage Accuracy:', average_accuracy_LSTM_1)
print('All Accuracies:', results["all_accuracies"])



Data loaded..


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Data pre-process completed..

Data pre-process completed..


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Maximum tweet length:  10
Total words:  12660

Model Summery ---------------------------------------------------------


...

79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9900 - loss: 0.0397
Epoch 4/5
78/79 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9917 - loss: 0.0313
Epoch 4: accuracy improved from 0.98899 to 0.98979, saving model to models/LSTM.h5


79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.9916 - loss: 0.0313
Epoch 5/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9918 - loss: 0.0293
Epoch 5: accuracy improved from 0.98979 to 0.99129, saving model to models/LSTM.h5


79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - accuracy: 0.9918 - loss: 0.0293


Testing ---------------------------------------------------------------------
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

Correct Prediction: 2568
Wrong Prediction: 763
Accuracy: 77.09396577604323


Run 5/5 ----------------------------------------------------


Training ----------------------------------------------------------------
Epoch 1/5
77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9866 - loss: 0.0403
Epoch 1: accuracy did not improve from 0.99129
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9866 - loss: 0.0403
Epoch 2/5
78/79 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9920 - loss: 0.0282
Epoch 2: accuracy did not improve from 0.99129
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9920 - loss: 0.0282
Epoch 3/5
77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9924 - loss: 0.0242
Epoch 3: accuracy did not improve from 0.99129
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accurac

79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.9914 - loss: 0.0277
Epoch 5/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9930 - loss: 0.0213
Epoch 5: accuracy improved from 0.99289 to 0.99299, saving model to models/LSTM.h5


79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.9930 - loss: 0.0213


Testing ---------------------------------------------------------------------
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

Correct Prediction: 2560
Wrong Prediction: 771
Accuracy: 76.85379765836086



Final Results -----------------------------------------------------

Average Accuracy: 78.33083158210749
All Accuracies: [81.11678174722306, 78.65505854097869, 77.93455418793155, 77.09396577604323, 76.85379765836086]
